# Simulation Recovery Diagnostics

This notebook is designed to make the recovery study easier to read and easier to connect to the parameters that matter in the grid run:

- true number of clusters $C$
- number of assessors $N$
- number of items $m$
- block density
- dispersion parameter $\theta$

It works with both completed runs and partially completed runs inside `simulation_recovery_runs/`.

The main outputs are:

- tidy trial-level and scenario-level tables
- average recovery summaries for each parameter level
- interaction heatmaps showing where combinations of parameters become difficult
- focused hard-slice vs easy-slice views over $(N, m)$
- failure tables showing where recovery drops below a chosen threshold

In [25]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

RUNS_DIR = Path('simulation_recovery_runs')
MAIN_METRICS = {
    'aligned_cluster_accuracy': 'Cluster Accuracy',
    'adjusted_rand_index': 'ARI',
    'weighted_normalized_kemeny_p_half': 'Normalized Kemeny',
    'weighted_same_block_f1': 'Block F1',
    'weighted_block_count_error': 'Block Count Error',
    'weighted_strict_inversions': 'Strict Inversions',
}

PARAM_SPECS = [
    ('n_clusters_true', 'True C'),
    ('n_assessors', 'N assessors'),
    ('n_items', 'm items'),
    ('block_density', 'Block density'),
    ('theta', 'θ'),
]

METRIC_SPECS = [
    ('aligned_cluster_accuracy', 'Cluster accuracy', False),
    ('adjusted_rand_index', 'ARI', False),
    ('weighted_same_block_f1', 'Block F1', False),
    ('weighted_normalized_kemeny_p_half', 'Order score', True),
]

PERFORMANCE_CMAP = 'RdYlGn'
BIAS_CMAP = 'RdBu_r'


def strip_seed(name: str) -> str:
    return name.rsplit('_seed', 1)[0] if '_seed' in name else name


def run_directories() -> list[Path]:
    if not RUNS_DIR.exists():
        return []
    return sorted([path for path in RUNS_DIR.iterdir() if path.is_dir()])


def latest_run() -> Path:
    runs = run_directories()
    if not runs:
        raise FileNotFoundError(f'No run directories found in {RUNS_DIR}')
    return runs[-1]


def load_run(run_dir: Path) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    metadata_path = run_dir / 'run_metadata.json'
    all_results_path = run_dir / 'all_results.json'

    metadata: dict[str, Any] = {}
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    metadata.setdefault('output_dir', str(run_dir))

    if all_results_path.exists():
        results = json.loads(all_results_path.read_text(encoding='utf-8'))
        return metadata, results

    per_scenario_files = sorted(
        [
            path for path in run_dir.glob('*.json')
            if path.name not in {'run_metadata.json', 'all_results.json', 'scenario_summary.json'}
        ]
    )
    results = [json.loads(path.read_text(encoding='utf-8')) for path in per_scenario_files]
    return metadata, results


def flatten_results(results: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for result in results:
        scenario = result['scenario']
        settings = result['settings']
        metrics = result['metrics']
        rows.append({
            'scenario_name': scenario['name'],
            'base_scenario': strip_seed(scenario['name']),
            'seed': scenario['seed'],
            'n_clusters_true': scenario['n_clusters'],
            'n_assessors': scenario['n_assessors'],
            'n_items': scenario['n_items'],
            'theta': scenario['theta'],
            'block_density': scenario.get('block_density'),
            'fit_n_clusters': settings.get('fit_n_clusters'),
            'n_iter': settings.get('n_iter'),
            'burn_in': settings.get('burn_in'),
            'thin': settings.get('thin'),
            'n_restarts': settings.get('n_restarts'),
            'runtime_seconds': result.get('runtime_seconds'),
            **metrics,
        })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(
        ['n_clusters_true', 'n_assessors', 'n_items', 'block_density', 'theta', 'seed']
    ).reset_index(drop=True)


def aggregate_trials(trial_df: pd.DataFrame) -> pd.DataFrame:
    if trial_df.empty:
        return pd.DataFrame()
    group_cols = [
        'base_scenario', 'n_clusters_true', 'n_assessors', 'n_items',
        'block_density', 'theta', 'fit_n_clusters', 'n_iter', 'burn_in',
        'thin', 'n_restarts'
    ]
    metric_cols = list(MAIN_METRICS.keys())
    agg_cols = metric_cols + ['n_pred_clusters', 'runtime_seconds']

    summary = trial_df.groupby(group_cols, dropna=False)[agg_cols].agg(['mean', 'std', 'count']).reset_index()
    summary.columns = [
        '_'.join([str(part) for part in col if part]).rstrip('_')
        if isinstance(col, tuple) else col
        for col in summary.columns
    ]
    return summary.sort_values(['n_clusters_true', 'n_assessors', 'n_items', 'theta']).reset_index(drop=True)


def metric_mean_col(metric_key: str) -> str:
    return f'{metric_key}_mean'


def metric_std_col(metric_key: str) -> str:
    return f'{metric_key}_std'


def _pretty_level(value: Any) -> str:
    if pd.isna(value):
        return 'NA'
    if isinstance(value, (int, np.integer)):
        return f'{int(value)}'
    if isinstance(value, (float, np.floating)):
        return f'{float(value):.2f}'.rstrip('0').rstrip('.')
    return str(value)


def _metric_values(df: pd.DataFrame, metric_key: str, invert: bool = False) -> pd.Series:
    values = df[metric_mean_col(metric_key)].astype(float)
    return 1.0 - values if invert else values


def show_parameter_grid(metadata: dict[str, Any], summary_df: pd.DataFrame) -> None:
    varied_rows = []
    for key, label in PARAM_SPECS:
        levels = sorted(summary_df[key].dropna().unique().tolist())
        varied_rows.append({
            'parameter': label,
            'levels': ', '.join(_pretty_level(v) for v in levels),
            'n_levels': len(levels),
        })

    print(f"Run: {Path(metadata.get('output_dir', run_dir)).name}")
    display(pd.DataFrame(varied_rows))

    fixed_settings = {
        'fit_n_clusters': summary_df['fit_n_clusters'].dropna().unique().tolist(),
        'n_iter': summary_df['n_iter'].dropna().unique().tolist(),
        'burn_in': summary_df['burn_in'].dropna().unique().tolist(),
        'thin': summary_df['thin'].dropna().unique().tolist(),
        'n_restarts': summary_df['n_restarts'].dropna().unique().tolist(),
    }
    fixed_settings = {k: (v[0] if len(v) == 1 else v) for k, v in fixed_settings.items() if len(v) > 0}
    print('Fixed fitting settings:', fixed_settings)


def compute_main_effects(summary_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for param_key, param_label in PARAM_SPECS:
        for level, group in summary_df.groupby(param_key, sort=True, dropna=False):
            for metric_key, metric_label, invert in METRIC_SPECS:
                values = _metric_values(group, metric_key, invert=invert)
                rows.append({
                    'parameter_key': param_key,
                    'parameter': param_label,
                    'level': level,
                    'level_label': _pretty_level(level),
                    'metric': metric_label,
                    'mean': values.mean(),
                    'se': values.std(ddof=1) / np.sqrt(len(values)) if len(values) > 1 else 0.0,
                    'n_scenarios': len(values),
                })
    return pd.DataFrame(rows)


def add_cluster_count_diagnostics(trial_df: pd.DataFrame, summary_df: pd.DataFrame) -> pd.DataFrame:
    group_cols = ['base_scenario', 'n_clusters_true', 'n_assessors', 'n_items', 'block_density', 'theta']
    if trial_df.empty:
        out = summary_df.copy()
        out['cluster_recovery_rate'] = np.nan
        out['cluster_count_abs_error'] = np.nan
        out['cluster_count_bias'] = np.nan
        return out

    tmp = trial_df.copy()
    tmp['cluster_match'] = (tmp['n_pred_clusters'] == tmp['n_clusters_true']).astype(float)
    tmp['cluster_count_abs_error'] = (tmp['n_pred_clusters'] - tmp['n_clusters_true']).abs()
    tmp['cluster_count_bias'] = tmp['n_pred_clusters'] - tmp['n_clusters_true']

    stats = (
        tmp.groupby(group_cols, dropna=False)
        .agg(
            cluster_recovery_rate=('cluster_match', 'mean'),
            cluster_count_abs_error=('cluster_count_abs_error', 'mean'),
            cluster_count_bias=('cluster_count_bias', 'mean'),
        )
        .reset_index()
    )
    return summary_df.merge(stats, on=group_cols, how='left')


def display_main_effect_tables(summary_df: pd.DataFrame) -> None:
    effects = compute_main_effects(summary_df)
    for _, param_label in PARAM_SPECS:
        table = (
            effects[effects['parameter'] == param_label]
            .pivot(index='level_label', columns='metric', values='mean')
            .round(3)
            .sort_index()
        )
        print(f'Average recovery by {param_label}')
        display(table)


def display_cluster_count_impact(cluster_df: pd.DataFrame) -> None:
    print('Recovery of the number of clusters')
    by_c = (
        cluster_df.groupby('n_clusters_true')[['n_pred_clusters_mean', 'cluster_recovery_rate', 'cluster_count_abs_error']]
        .mean()
        .rename(columns={
            'n_pred_clusters_mean': 'mean predicted clusters',
            'cluster_recovery_rate': 'exact C recovery rate',
            'cluster_count_abs_error': 'mean |Ĉ - C|',
        })
        .round(3)
    )
    display(by_c)

    impact = (
        cluster_df.groupby('cluster_count_abs_error')[[
            metric_mean_col('aligned_cluster_accuracy'),
            metric_mean_col('adjusted_rand_index'),
            metric_mean_col('weighted_same_block_f1'),
            metric_mean_col('weighted_normalized_kemeny_p_half'),
        ]]
        .mean()
        .reset_index()
        .sort_values('cluster_count_abs_error')
    )
    impact['order_score_mean'] = 1.0 - impact[metric_mean_col('weighted_normalized_kemeny_p_half')]
    impact = impact.rename(columns={
        'cluster_count_abs_error': 'mean |Ĉ - C|',
        metric_mean_col('aligned_cluster_accuracy'): 'cluster accuracy',
        metric_mean_col('adjusted_rand_index'): 'ARI',
        metric_mean_col('weighted_same_block_f1'): 'Block F1',
        'order_score_mean': 'order score',
    })
    print('Performance conditional on cluster-count error')
    display(impact[['mean |Ĉ - C|', 'cluster accuracy', 'ARI', 'Block F1', 'order score']].round(3))


def plot_main_effects(summary_df: pd.DataFrame) -> None:
    effects = compute_main_effects(summary_df)
    metric_order = [label for _, label, _ in METRIC_SPECS]
    colors = sns.color_palette('deep', len(metric_order))

    fig, axes = plt.subplots(len(metric_order), len(PARAM_SPECS), figsize=(18, 10), constrained_layout=True)

    for col_idx, (_, param_label) in enumerate(PARAM_SPECS):
        subset = effects[effects['parameter'] == param_label].copy()
        for row_idx, metric_label in enumerate(metric_order):
            ax = axes[row_idx, col_idx]
            metric_df = subset[subset['metric'] == metric_label].sort_values('level')
            x = np.arange(len(metric_df))
            ax.errorbar(
                x,
                metric_df['mean'],
                yerr=1.96 * metric_df['se'],
                marker='o',
                linewidth=2.2,
                capsize=3,
                color=colors[row_idx],
            )
            ax.set_xticks(x)
            ax.set_xticklabels(metric_df['level_label'])
            ax.set_ylim(0, 1.02)
            ax.grid(alpha=0.2, linestyle=':')
            if row_idx == 0:
                ax.set_title(param_label, fontweight='bold')
            if col_idx == 0:
                ax.set_ylabel(metric_label)

    fig.suptitle('Average recovery at each parameter level (mean ± 95% CI over the remaining grid)', fontsize=14, fontweight='bold')
    plt.show()


def _pairwise_matrix(summary_df: pd.DataFrame, row_key: str, col_key: str, metric_key: str, invert: bool = False) -> pd.DataFrame:
    values = summary_df[[row_key, col_key, metric_mean_col(metric_key)]].copy()
    values['score'] = _metric_values(values, metric_key, invert=invert)
    return (
        values.pivot_table(index=row_key, columns=col_key, values='score', aggfunc='mean')
        .sort_index()
        .sort_index(axis=1)
    )


def plot_interaction_heatmaps(summary_df: pd.DataFrame) -> None:
    pair_specs = [
        ('n_clusters_true', 'theta', 'C × θ'),
        ('n_clusters_true', 'n_items', 'C × m'),
        ('theta', 'block_density', 'θ × block density'),
        ('n_assessors', 'n_items', 'N × m'),
    ]
    metric_rows = [
        ('aligned_cluster_accuracy', 'Cluster accuracy', False),
        ('weighted_normalized_kemeny_p_half', 'Order score', True),
    ]
    label_map = {key: label for key, label in PARAM_SPECS}

    fig, axes = plt.subplots(len(metric_rows), len(pair_specs), figsize=(18, 8), constrained_layout=True)

    for row_idx, (metric_key, metric_label, invert) in enumerate(metric_rows):
        for col_idx, (row_key, col_key, title) in enumerate(pair_specs):
            ax = axes[row_idx, col_idx]
            pivot = _pairwise_matrix(summary_df, row_key, col_key, metric_key, invert=invert)
            sns.heatmap(
                pivot,
                annot=True,
                fmt='.2f',
                cmap=PERFORMANCE_CMAP,
                vmin=0,
                vmax=1,
                cbar=(col_idx == len(pair_specs) - 1),
                ax=ax,
            )
            ax.set_title(f'{metric_label}: {title}', fontsize=11, fontweight='bold')
            ax.set_xlabel(label_map.get(col_key, col_key))
            ax.set_ylabel(label_map.get(row_key, row_key))

    plt.show()


def plot_cluster_count_diagnostics(cluster_df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)

    rate_ct = cluster_df.pivot_table(index='n_clusters_true', columns='theta', values='cluster_recovery_rate', aggfunc='mean').sort_index().sort_index(axis=1)
    sns.heatmap(rate_ct, annot=True, fmt='.2f', cmap=PERFORMANCE_CMAP, vmin=0, vmax=1, ax=axes[0, 0], cbar=False)
    axes[0, 0].set_title('Exact recovery rate of Ĉ: C × θ', fontweight='bold')
    axes[0, 0].set_xlabel('θ')
    axes[0, 0].set_ylabel('True C')

    rate_cm = cluster_df.pivot_table(index='n_clusters_true', columns='n_items', values='cluster_recovery_rate', aggfunc='mean').sort_index().sort_index(axis=1)
    sns.heatmap(rate_cm, annot=True, fmt='.2f', cmap=PERFORMANCE_CMAP, vmin=0, vmax=1, ax=axes[0, 1], cbar=False)
    axes[0, 1].set_title('Exact recovery rate of Ĉ: C × m', fontweight='bold')
    axes[0, 1].set_xlabel('m items')
    axes[0, 1].set_ylabel('True C')

    bias_ct = cluster_df.pivot_table(index='n_clusters_true', columns='theta', values='cluster_count_bias', aggfunc='mean').sort_index().sort_index(axis=1)
    sns.heatmap(bias_ct, annot=True, fmt='.2f', cmap=BIAS_CMAP, center=0, ax=axes[0, 2], cbar=False)
    axes[0, 2].set_title('Mean cluster-count bias: Ĉ − C', fontweight='bold')
    axes[0, 2].set_xlabel('θ')
    axes[0, 2].set_ylabel('True C')

    sns.scatterplot(
        data=cluster_df,
        x='cluster_count_abs_error',
        y=metric_mean_col('aligned_cluster_accuracy'),
        hue='n_clusters_true',
        size='n_items',
        palette='deep',
        alpha=0.8,
        ax=axes[1, 0],
    )
    axes[1, 0].set_title('Accuracy vs |Ĉ − C|', fontweight='bold')
    axes[1, 0].set_xlabel('Mean absolute cluster-count error')
    axes[1, 0].set_ylabel('Cluster accuracy')
    axes[1, 0].set_ylim(0, 1.02)

    sns.scatterplot(
        data=cluster_df,
        x='cluster_count_abs_error',
        y=metric_mean_col('adjusted_rand_index'),
        hue='n_clusters_true',
        size='n_items',
        palette='deep',
        legend=False,
        alpha=0.8,
        ax=axes[1, 1],
    )
    axes[1, 1].set_title('ARI vs |Ĉ − C|', fontweight='bold')
    axes[1, 1].set_xlabel('Mean absolute cluster-count error')
    axes[1, 1].set_ylabel('ARI')
    axes[1, 1].set_ylim(0, 1.02)

    perf_by_error = (
        cluster_df.groupby('cluster_count_abs_error')[metric_mean_col('weighted_same_block_f1')]
        .mean()
        .reset_index()
        .sort_values('cluster_count_abs_error')
    )
    axes[1, 2].plot(perf_by_error['cluster_count_abs_error'], perf_by_error[metric_mean_col('weighted_same_block_f1')], marker='o', linewidth=2.5)
    axes[1, 2].set_title('Block F1 vs |Ĉ − C|', fontweight='bold')
    axes[1, 2].set_xlabel('Mean absolute cluster-count error')
    axes[1, 2].set_ylabel('Block F1')
    axes[1, 2].set_ylim(0, 1.02)
    axes[1, 2].grid(alpha=0.2, linestyle=':')

    plt.show()


def plot_theta_density_slices(summary_df: pd.DataFrame) -> None:
    metric_key = 'aligned_cluster_accuracy'
    metric_col = metric_mean_col(metric_key)
    grouped = (
        summary_df.groupby(['n_clusters_true', 'block_density', 'theta'])[metric_col]
        .mean()
        .reset_index()
        .sort_values(['n_clusters_true', 'block_density', 'theta'])
    )
    clusters = sorted(grouped['n_clusters_true'].unique())
    fig, axes = plt.subplots(1, len(clusters), figsize=(5 * len(clusters), 4), sharey=True, constrained_layout=True)
    if len(clusters) == 1:
        axes = [axes]

    palette = sns.color_palette('rocket_r', grouped['block_density'].nunique())
    for ax, cluster_count in zip(axes, clusters):
        subset = grouped[grouped['n_clusters_true'] == cluster_count]
        for color, (density, group) in zip(palette, subset.groupby('block_density')):
            ax.plot(group['theta'], group[metric_col], marker='o', linewidth=2.5, color=color, label=f'bd={density:.2f}')
        ax.set_title(f'Accuracy vs θ by block density\nTrue C={cluster_count}', fontweight='bold')
        ax.set_xlabel('θ')
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.2, linestyle=':')
        if ax is axes[0]:
            ax.set_ylabel('Cluster accuracy')
        ax.legend(title='Block density', fontsize=9)
    plt.show()


def plot_extreme_nm_slices(summary_df: pd.DataFrame, metric_key: str = 'aligned_cluster_accuracy') -> None:
    metric_col = metric_mean_col(metric_key)
    hardness = (
        summary_df.groupby(['n_clusters_true', 'theta', 'block_density'])[metric_col]
        .mean()
        .reset_index()
    )
    hardest = hardness.nsmallest(1, metric_col).iloc[0]
    easiest = hardness.nlargest(1, metric_col).iloc[0]

    scenarios = [('Hardest average slice', hardest), ('Easiest average slice', easiest)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

    for ax, (title, row) in zip(axes, scenarios):
        subset = summary_df[
            (summary_df['n_clusters_true'] == row['n_clusters_true'])
            & (summary_df['theta'] == row['theta'])
            & (summary_df['block_density'] == row['block_density'])
        ]
        pivot = (
            subset.pivot_table(index='n_items', columns='n_assessors', values=metric_col, aggfunc='mean')
            .sort_index()
            .sort_index(axis=1)
        )
        sns.heatmap(pivot, annot=True, fmt='.2f', cmap=PERFORMANCE_CMAP, vmin=0, vmax=1, cbar=(ax is axes[-1]), ax=ax)
        ax.set_title(
            f"{title}\nC={int(row['n_clusters_true'])}, θ={row['theta']:.1f}, bd={row['block_density']:.2f}",
            fontweight='bold',
        )
        ax.set_xlabel('N assessors')
        ax.set_ylabel('m items')

    plt.show()


def failure_table(summary_df: pd.DataFrame, metric_key: str = 'aligned_cluster_accuracy', threshold: float = 0.8) -> pd.DataFrame:
    metric_col = metric_mean_col(metric_key)
    cols = [
        'base_scenario', 'n_clusters_true', 'n_assessors', 'n_items', 'block_density', 'theta',
        metric_col,
        metric_mean_col('adjusted_rand_index'),
        metric_mean_col('weighted_normalized_kemeny_p_half'),
        metric_mean_col('weighted_same_block_f1'),
        metric_mean_col('weighted_block_count_error'),
    ]
    return summary_df.loc[summary_df[metric_col] < threshold, cols].sort_values(
        ['n_clusters_true', 'block_density', 'theta', 'n_assessors', 'n_items', metric_col]
    )

In [26]:
RUN_NAME = 'recovery_contrast_20260415_163102'
# Set RUN_NAME = None to automatically use the newest run directory.

run_dir = RUNS_DIR / RUN_NAME if RUN_NAME else latest_run()
metadata, results = load_run(run_dir)
trial_df = flatten_results(results)
summary_df = aggregate_trials(trial_df)

print(f'Run directory: {run_dir}')
print(f'Loaded {len(results)} result file(s)')
print(f'Trial rows: {len(trial_df)}')
print(f'Scenario rows: {len(summary_df)}')
print(json.dumps(metadata, indent=2))

summary_cols = [
    'base_scenario',
    'n_clusters_true',
    'n_assessors',
    'n_items',
    'block_density',
    'theta',
    'n_pred_clusters_mean',
    metric_mean_col('aligned_cluster_accuracy'),
    metric_mean_col('adjusted_rand_index'),
    metric_mean_col('weighted_normalized_kemeny_p_half'),
    metric_mean_col('weighted_same_block_f1'),
    metric_mean_col('weighted_block_count_error'),
    'runtime_seconds_mean',
    'aligned_cluster_accuracy_count',
]

trial_df.head()

IndentationError: expected an indented block after 'if' statement on line 64 (2239111151.py, line 65)

In [20]:
display(summary_df[summary_cols].round(3))

print('Best scenarios by accuracy')
display(summary_df.sort_values(metric_mean_col('aligned_cluster_accuracy'), ascending=False)[summary_cols].head(10).round(3))

print('Worst scenarios by accuracy')
display(summary_df.sort_values(metric_mean_col('aligned_cluster_accuracy'), ascending=True)[summary_cols].head(10).round(3))

cluster_df = add_cluster_count_diagnostics(trial_df, summary_df)

show_parameter_grid(metadata, summary_df)
display_main_effect_tables(summary_df)
display_cluster_count_impact(cluster_df)
plot_main_effects(summary_df)
plot_cluster_count_diagnostics(cluster_df)

Run directory: simulation_recovery_runs\recovery_early_20260414_175404
Loaded 6 result file(s)
Trial rows: 6
Scenario rows: 3
{
  "mode": "early",
  "n_iter": 8000,
  "burn_in": 5000,
  "thin": 20,
  "n_restarts": 3,
  "n_trials": 6,
  "n_base_scenarios": 3,
  "base_scenarios": [
    "c2_n200_m12_theta0p8",
    "c3_n300_m15_theta1p5",
    "c4_n400_m18_theta2p5"
  ],
  "output_dir": "simulation_recovery_runs\\recovery_early_20260414_175404",
  "created_at": "20260414_175404"
}


,scenario_name,base_scenario,seed,n_clusters_true,n_assessors,n_items,theta,fit_n_clusters,n_iter,burn_in,thin,n_restarts,runtime_seconds,aligned_cluster_accuracy,adjusted_rand_index,weighted_kemeny_p_half,weighted_normalized_kemeny_p_half,weighted_strict_inversions,weighted_same_block_f1,weighted_block_count_error,n_true_clusters,n_pred_clusters,cluster_mapping_pred_to_true,confusion,per_cluster
0,c2_n200_m12_theta0p8_seed11,c2_n200_m12_theta0p8,11,2,200,12,0.8,4,8000,5000,20,3,41.554620,1.00,1.000000,0.000,0.000000,0.000000,1.000000,0.000,2,4,"{'3': 0, '1': 1}","[[0, 0, 0, 110], [0, 90, 0, 0]]","[{'true_cluster': 0, 'matched_pred_cluster': 3..."
1,c2_n200_m12_theta0p8_seed17,c2_n200_m12_theta0p8,17,2,200,12,0.8,4,8000,5000,20,3,40.528780,1.00,1.000000,0.000,0.000000,0.000000,1.000000,0.000,2,4,"{'3': 0, '2': 1}","[[0, 0, 0, 101], [0, 0, 99, 0]]","[{'true_cluster': 0, 'matched_pred_cluster': 3..."
2,c3_n300_m15_theta1p5_seed11,c3_n300_m15_theta1p5,11,3,300,15,1.5,6,8000,5000,20,3,40.315352,1.00,1.000000,0.000,0.000000,0.000000,1.000000,0.000,3,6,"{'2': 0, '0': 1, '4': 2}","[[0, 0, 105, 0, 0, 0], [106, 0, 0, 0, 0, 0], [...","[{'true_cluster': 0, 'matched_pred_cluster': 2..."
3,c3_n300_m15_theta1p5_seed17,c3_n300_m15_theta1p5,17,3,300,15,1.5,6,8000,5000,20,3,29.402763,0.67,0.541302,23.965,0.228238,14.256667,0.449058,3.730,3,6,"{'2': 0, '0': 1, '1': 2}","[[99, 0, 0, 0, 0, 0], [109, 0, 0, 0, 0, 0], [0...","[{'true_cluster': 0, 'matched_pred_cluster': 2..."
4,c4_n400_m18_theta2p5_seed11,c4_n400_m18_theta2p5,11,4,400,18,2.5,8,8000,5000,20,3,21.522627,0.51,0.329304,46.780,0.305752,30.865000,0.354135,4.715,4,8,"{'0': 0, '3': 1, '6': 2, '1': 3}","[[0, 0, 0, 102, 0, 0, 0, 0], [0, 0, 0, 105, 0,...","[{'true_cluster': 0, 'matched_pred_cluster': 0..."


In [21]:
plot_interaction_heatmaps(summary_df)
plot_theta_density_slices(summary_df)
plot_extreme_nm_slices(summary_df)

BREAK_THRESHOLD = 0.8
break_df = failure_table(summary_df, metric_key='aligned_cluster_accuracy', threshold=BREAK_THRESHOLD)

print(f'Scenarios with mean accuracy below {BREAK_THRESHOLD:.2f}')
display(break_df.round(3))

KeyError: "['block_density'] not in index"

## Follow-up stress test: rerun the hardest scenarios

This section re-runs the worst scenarios from the current simulation output using a longer chain and annealing during burn-in. The goal is to check whether some of the poor recovery was due to insufficient mixing rather than intrinsic difficulty.

In [ ]:
from _simulation_recovery_study import Scenario, run_scenario


def build_followup_scenario(row: pd.Series, trial_df: pd.DataFrame) -> Scenario:
    seed = int(trial_df.loc[trial_df['base_scenario'] == row['base_scenario'], 'seed'].iloc[0])
    return Scenario(
        name=f"{row['base_scenario']}_followup_seed{seed}",
        n_clusters=int(row['n_clusters_true']),
        n_assessors=int(row['n_assessors']),
        n_items=int(row['n_items']),
        theta=float(row['theta']),
        block_density=float(row['block_density']),
        seed=seed,
    )


def rerun_worst_scenarios(
    summary_df: pd.DataFrame,
    trial_df: pd.DataFrame,
    *,
    n_cases: int = 4,
    metric_key: str = 'aligned_cluster_accuracy',
    n_iter: int = 16000,
    burn_in: int = 10000,
    thin: int = 20,
    n_restarts: int = 4,
    use_annealing: bool = True,
    temp_min: float = 0.1,
    temp_max: float = 1.0,
) -> pd.DataFrame:
    baseline_col = metric_mean_col(metric_key)
    worst = summary_df.nsmallest(n_cases, baseline_col).copy()
    comparison_rows = []

    print(
        f'Rerunning {len(worst)} worst scenarios with n_iter={n_iter}, '
        f'burn_in={burn_in}, annealing={use_annealing}, temp_min={temp_min}'
    )

    for i, (_, row) in enumerate(worst.iterrows(), start=1):
        scenario = build_followup_scenario(row, trial_df)
        print(f'[{i}/{len(worst)}] {scenario.name}')
        rerun = run_scenario(
            scenario,
            n_iter=n_iter,
            burn_in=burn_in,
            thin=thin,
            n_restarts=n_restarts,
            use_annealing=use_annealing,
            temp_min=temp_min,
            temp_max=temp_max,
        )
        m = rerun['metrics']
        comparison_rows.append({
            'scenario': row['base_scenario'],
            'C': int(row['n_clusters_true']),
            'N': int(row['n_assessors']),
            'm_items': int(row['n_items']),
            'block_density': float(row['block_density']),
            'theta': float(row['theta']),
            'baseline_accuracy': float(row[metric_mean_col('aligned_cluster_accuracy')]),
            'rerun_accuracy': float(m['aligned_cluster_accuracy']),
            'baseline_ari': float(row[metric_mean_col('adjusted_rand_index')]),
            'rerun_ari': float(m['adjusted_rand_index']),
            'baseline_block_f1': float(row[metric_mean_col('weighted_same_block_f1')]),
            'rerun_block_f1': float(m['weighted_same_block_f1']),
            'baseline_order_score': 1.0 - float(row[metric_mean_col('weighted_normalized_kemeny_p_half')]),
            'rerun_order_score': 1.0 - float(m['weighted_normalized_kemeny_p_half']),
            'baseline_pred_clusters': float(row['n_pred_clusters_mean']),
            'rerun_pred_clusters': float(m['n_pred_clusters']),
            'runtime_seconds': float(rerun['runtime_seconds']),
        })

    compare_df = pd.DataFrame(comparison_rows)
    for metric in ['accuracy', 'ari', 'block_f1', 'order_score']:
        compare_df[f'delta_{metric}'] = compare_df[f'rerun_{metric}'] - compare_df[f'baseline_{metric}']
    compare_df['delta_pred_clusters'] = compare_df['rerun_pred_clusters'] - compare_df['baseline_pred_clusters']
    return compare_df.sort_values('delta_accuracy', ascending=False)


def plot_followup_comparison(compare_df: pd.DataFrame) -> None:
    metric_specs = [
        ('accuracy', 'Cluster accuracy'),
        ('ari', 'ARI'),
        ('block_f1', 'Block F1'),
        ('order_score', 'Order score'),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)

    for ax, (metric_key, label) in zip(axes.flat, metric_specs):
        for _, row in compare_df.iterrows():
            ax.plot([0, 1], [row[f'baseline_{metric_key}'], row[f'rerun_{metric_key}']], marker='o', linewidth=2)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Original run', 'Longer + annealed'])
        ax.set_ylim(0, 1.02)
        ax.set_title(label, fontweight='bold')
        ax.grid(alpha=0.2, linestyle=':')

    fig.suptitle('Worst-case follow-up: does a longer annealed run help?', fontsize=14, fontweight='bold')
    plt.show()


followup_df = rerun_worst_scenarios(
    summary_df,
    trial_df,
    n_cases=4,
    n_iter=16000,
    burn_in=10000,
    thin=20,
    n_restarts=4,
    use_annealing=True,
    temp_min=0.1,
    temp_max=1.0,
)

followup_display = followup_df[[
    'scenario', 'C', 'N', 'm_items', 'block_density', 'theta',
    'baseline_accuracy', 'rerun_accuracy', 'delta_accuracy',
    'baseline_ari', 'rerun_ari', 'delta_ari',
    'baseline_block_f1', 'rerun_block_f1', 'delta_block_f1',
    'baseline_pred_clusters', 'rerun_pred_clusters', 'delta_pred_clusters',
    'runtime_seconds',
]].round(3)

display(followup_display)
print('Mean deltas across rerun cases:')
display(followup_df[[c for c in followup_df.columns if c.startswith('delta_')]].mean().to_frame('mean_delta').T.round(3))
plot_followup_comparison(followup_df)

## Ablation: annealing alone

This check keeps the original simulation budget and only switches annealing on. That isolates whether the gain came from annealing itself rather than from the longer run.

In [ ]:
def summarize_ablation(name: str, df: pd.DataFrame) -> dict[str, float]:
    return {
        'setting': name,
        'mean Δ accuracy': df['delta_accuracy'].mean(),
        'mean Δ ARI': df['delta_ari'].mean(),
        'mean Δ Block F1': df['delta_block_f1'].mean(),
        'mean Δ order score': df['delta_order_score'].mean(),
        'mean Δ predicted clusters': df['delta_pred_clusters'].mean(),
    }


def plot_ablation_summary(ablation_df: pd.DataFrame) -> None:
    metric_cols = ['mean Δ accuracy', 'mean Δ ARI', 'mean Δ Block F1', 'mean Δ order score']
    plot_df = ablation_df.set_index('setting')[metric_cols]
    ax = plot_df.plot(kind='bar', figsize=(10, 5), ylim=(0, 1.05), rot=0)
    ax.set_ylabel('Average improvement over original run')
    ax.set_title('How much improvement comes from annealing alone?', fontweight='bold')
    ax.grid(axis='y', alpha=0.2, linestyle=':')
    plt.tight_layout()
    plt.show()


base_n_iter = int(summary_df['n_iter'].iloc[0])
base_burn_in = int(summary_df['burn_in'].iloc[0])
base_thin = int(summary_df['thin'].iloc[0])
base_restarts = int(summary_df['n_restarts'].iloc[0])

anneal_only_df = rerun_worst_scenarios(
    summary_df,
    trial_df,
    n_cases=4,
    n_iter=base_n_iter,
    burn_in=base_burn_in,
    thin=base_thin,
    n_restarts=base_restarts,
    use_annealing=True,
    temp_min=0.1,
    temp_max=1.0,
)

anneal_only_display = anneal_only_df[[
    'scenario', 'baseline_accuracy', 'rerun_accuracy', 'delta_accuracy',
    'baseline_ari', 'rerun_ari', 'delta_ari',
    'baseline_block_f1', 'rerun_block_f1', 'delta_block_f1',
    'baseline_pred_clusters', 'rerun_pred_clusters', 'delta_pred_clusters',
]].round(3)

print('Annealing only, keeping the original iteration budget')
display(anneal_only_display)

ablation_df = pd.DataFrame([
    summarize_ablation('Annealing only', anneal_only_df),
    summarize_ablation('Longer + annealed', followup_df),
]).round(3)

print('Average gain across the same worst scenarios')
display(ablation_df)
plot_ablation_summary(ablation_df)

## Iteration sensitivity on a deliberately hard dataset

This section fixes one difficult synthetic scenario and re-fits the model over a range of iteration budgets. The goal is to identify the point after which the performance is essentially unchanged.